# BBC RAG Assistant
# 01 – Problem Framing & Data Understanding


## Project title
Enterprise Document Intelligence Assistant (BBC News) using LLM + RAG

## Goal
Build an assistant that:
- Summarizes BBC news articles,
- Answers questions grounded in the article text,
- Supports conversational exploration (follow-up questions and comparisons),

using embeddings + vector search (RAG) with a Large Language Model (LLM) on top.

This notebook focuses only on:
- Loading and understanding the BBC dataset,
- Defining a clean document schema,
- Writing down user journeys, question types, success criteria, and a high-level architecture.



In [1]:
# Importing necessary libraries
import os
import pandas as pd
from sklearn.datasets import load_files
import pandas as pd

pd.set_option("display.max_colwidth", 300)


In [2]:

# Point to the 'bbc' folder, which contains the category subfolders
data_path = "/kaggle/input/datasets/shivamkushwaha/bbc-full-text-document-classification/bbc"

# Load the files
data = load_files(data_path, encoding="latin-1", decode_error="replace")

# Create the DataFrame
df = pd.DataFrame({
    'text': data.data,
    'target': data.target,
    'category': [data.target_names[t] for t in data.target]
})

print(df.head())

                                                                                                                                                                                                                                                                                                          text  \
0  Tate & Lyle boss bags top award\n\nTate & Lyle's chief executive has been named European Businessman of the Year by a leading business magazine.\n\nIain Ferguson was awarded the title by US publication Forbes for returning one of the UK's "venerable" manufacturers to the country's top 100 compan...   
1  Halo 2 sells five million copies\n\nMicrosoft is celebrating bumper sales of its Xbox sci-fi shooter, Halo 2.\n\nThe game has sold more than five million copies worldwide since it went on sale in mid-November, the company said. Halo 2 has proved popular online, with gamers notching up a record 2...   
2  MSPs hear renewed climate warning\n\nClimate change could be completely out of 

In [3]:
# Reset index to create a stable doc_id
df = df.reset_index(drop=True)
df["doc_id"] = df.index

# Create a synthetic title from the first line / first characters
def make_title(text, max_len=80):
    if not isinstance(text, str):
        return ""
    first_line = text.strip().split("\n")[0]
    if len(first_line) > max_len:
        return first_line[:max_len] + "..."
    return first_line

df["title"] = df["text"].apply(make_title)

# Keep only what you need for the RAG project
df_standard = df[["doc_id", "title", "category", "text"]].copy()

df_standard.head()


,doc_id,title,category,text
0,0,Tate & Lyle boss bags top award,business,"Tate & Lyle boss bags top award\n\nTate & Lyle's chief executive has been named European Businessman of the Year by a leading business magazine.\n\nIain Ferguson was awarded the title by US publication Forbes for returning one of the UK's ""venerable"" manufacturers to the country's top 100 compan..."
1,1,Halo 2 sells five million copies,tech,"Halo 2 sells five million copies\n\nMicrosoft is celebrating bumper sales of its Xbox sci-fi shooter, Halo 2.\n\nThe game has sold more than five million copies worldwide since it went on sale in mid-November, the company said. Halo 2 has proved popular online, with gamers notching up a record 2..."
2,2,MSPs hear renewed climate warning,politics,"MSPs hear renewed climate warning\n\nClimate change could be completely out of control within several decades, the Scottish Environment Protection Agency is warning a committee of MSPs.\n\nExperts are giving evidence on the subject to the Scottish Parliament's environment committee. Officials be..."
3,3,Pavey focuses on indoor success,sport,Pavey focuses on indoor success\n\nJo Pavey will miss January's View From Great Edinburgh International Cross Country to focus on preparing for the European Indoor Championships in March.\n\nThe 31-year-old was third behind Hayley Yelling and Justyna Bak in last week's European Cross Country Cha...
4,4,Tories reject rethink on axed MP,politics,"Tories reject rethink on axed MP\n\nSacked MP Howard Flight's local Conservative association has insisted he will not be its candidate at the general election.\n\nRussell Tanguay, agent for Arundel and South Downs Tories, said Mr Flight was ineligible to be a candidate and the association was se..."


## Standard document schema

From this point onward, each BBC article will be represented as:

- `doc_id`: integer ID (row index after reset)
- `title`: short title synthesized from the article's first line / first characters
- `category`: news topic (e.g., business, entertainment, politics, sport, tech)
- `text`: full article body (raw for now; will be cleaned and chunked later)

One row in `df_standard` = one document in the RAG system.


In [4]:
# Basis Stats
print("Number of articles:", len(df_standard))
print("\n Category distribution:")
print(df_standard["category"].value_counts())


Number of articles: 2225

 Category distribution:
category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


In [5]:
# Length Distribution
df_standard["text_length_chars"] = df_standard["text"].astype(str).str.len()
df_standard["text_length_words"] = df_standard["text"].astype(str).str.split().str.len()
df_standard[["text_length_chars", "text_length_words"]].describe()


,text_length_chars,text_length_words
count,2225.000000,2225.000000
mean,2265.790562,384.040449
std,1364.305951,238.174497
min,503.000000,89.000000
25%,1448.000000,246.000000
50%,1967.000000,332.000000
75%,2804.000000,471.000000
max,25485.000000,4432.000000


In [6]:
for i in range(3):
    row = df_standard.iloc[i]
    print("=" * 80)
    print(f"doc_id: {row['doc_id']}")
    print(f"category: {row['category']}")
    print(f"title: {row['title']}\n")
    print("TEXT PREVIEW:\n")
    print(row["text"][:800], "...\n")


doc_id: 0
category: business
title: Tate & Lyle boss bags top award

TEXT PREVIEW:

Tate & Lyle boss bags top award

Tate & Lyle's chief executive has been named European Businessman of the Year by a leading business magazine.

Iain Ferguson was awarded the title by US publication Forbes for returning one of the UK's "venerable" manufacturers to the country's top 100 companies. The sugar group had been absent from the FTSE 100 for seven years until Mr Ferguson helped it return to growth. Tate's shares have leapt 55% this year, boosted by firming sugar prices and sales of its artificial sweeteners.

"After years of a sagging stock price and a seven-year hiatus from the FTSE 100, one of Britain's venerable manufacturers has returned to the vaunted index," Forbes said. Mr Ferguson took the helm at the company in 2003, after spending most of his career at consumer goods giant ...

doc_id: 1
category: tech
title: Halo 2 sells five million copies

TEXT PREVIEW:

Halo 2 sells five million cop

In [7]:
# Save the standardized docs for reuse
df_standard.to_csv("bbc_docs_standard.csv", index=False)


## Initial data understanding

Based on the statistics:

- Number of articles: (auto-filled from the previous cell)
- Categories and counts: as printed above (e.g., business, entertainment, politics, sport, tech).
- Article length:
  - Characters and word counts: see the descriptive stats.
  - Each row is a reasonably long news article, suitable as a "document" for summarization and QA.

Conclusion: This dataset is appropriate as a corpus for a document intelligence assistant.
Each document is an individual BBC news article with a topic label.



## User journeys

1. **Single-article exploration**
   - User selects an article by `doc_id` or `title`.
   - The assistant should:
     - Provide a concise summary.
     - Answer detailed questions about that specific article.
     - Optionally produce a mini-report (main topic, key events, actors, outcome, risks).

2. **Multi-article / topic exploration**
   - User selects a `category` (e.g., politics) or provides a topic keyword (e.g., "Brexit").
   - The assistant should:
     - Summarize common themes across relevant articles.
     - Answer questions that span multiple documents (e.g., "How is Brexit portrayed?").

3. **Conversational exploration**
   - User interacts with a chatbot:
     - Start: "Summarize this article."
     - Follow-ups: "Explain in simpler terms", "Give me bullet points", "Compare with another article".
   - The assistant keeps context and refines answers over multiple turns.


## Question types to support

**Single-article questions**
- What is the main topic of this article?
- What key events are described?
- Who are the main people or organizations involved?
- What is the outcome or current status?
- Are any risks, controversies, or criticisms mentioned?
- What is the overall tone of the article?

**Multi-article / topic-level questions**
- What are the main themes in recent politics articles?
- How do different articles describe the same event?
- What recurring concerns are mentioned about topic X?

**Chat refinement questions**
- Explain that in simpler terms.
- Give me only bullet points.
- Compare this article with the previous one we discussed.


## Success criteria

**Summarization**
- Summaries are faithful to the article (no invented facts).
- Summaries are concise (e.g., 5–7 bullet points).
- They cover the main topic, key events, key actors, and outcome/status.

**Question answering**
- Answers are supported by the text of the article(s).
- If the required information is not present, the assistant clearly says it cannot answer from the given articles.
- Answers may reference the relevant article (e.g., by title or a short quote).

**Chatbot behavior**
- Coherent over multiple turns.
- Respects style instructions (simpler language, more detail, bullet points).
- Does not hallucinate information beyond the BBC article corpus when answering about it.


## Planned architecture (for later tasks)

- **Documents**
  - Use `df_standard` as the base corpus: each row is a document with {doc_id, title, category, text}.

- **Preprocessing (next task)**
  - Clean the text (remove noise/extra whitespace).
  - Split each article into overlapping chunks with metadata (doc_id, category, position).

- **Embeddings & vector index**
  - Compute embeddings for each chunk using a sentence embedding model.
  - Store embeddings + metadata in a vector index for semantic search.

- **RAG pipeline**
  - For summarization and QA:
    - Retrieve top-k relevant chunks for a query (article-specific or topic-level).
    - Pass retrieved chunks + query into an LLM.
    - Instruct the LLM to answer only using the provided context and to say "I don't know" when needed.

- **LLM**
  - Used for:
    - Single-article and multi-article summarization.
    - Question answering over retrieved chunks.
    - Conversational responses with a simple memory mechanism.

- **Interfaces**
  - Initially: notebook cells for exploration.
  - Later: simple API endpoints (e.g., `/summarize`, `/qa`, `/chat`) and possibly a lightweight UI.
